# Living Canvas: real-model test

Does an image model, steered by a building-up **crowd embedding**, stay one recognizable painting while each visitor's change stays visible?

**Loop:** each visitor's expression → CLIP text embedding → pulls the crowd vector `C` → `C` (blended with a fixed Monet style anchor) is passed to the model as `prompt_embeds` → image-to-image from the previous frame.

**Three runs over the same 10 example visitors as the web demo:**
1. **Raw words:** embed exactly what people said.
2. **Visual reading:** embed a short visual description of it (what Claude would produce).
3. **Text prompt only (baseline):** a plain text prompt each frame, no embedding math.

**Pass criteria. Judge these from the grids at the end:**
1. All 11 frames are clearly the same chapel and pond.
2. You can tell which visitor caused each change.
3. It looks like a painting, not mush. Watch for drift over repeated passes.
4. Under about 3 s per frame.

**Setup:** in Colab, *Runtime → Change runtime type → T4 GPU*. Use a personal or Duke Google account. Run the cells top to bottom. The smoke test (one frame) runs first.

In [ ]:
!pip -q install "diffusers==0.31.0" "transformers==4.46.3" "accelerate==1.1.1"

In [ ]:
import torch, time, os
assert torch.cuda.is_available(), "No GPU found. In Colab: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

## Load SD-Turbo (about 2.5 GB, a minute or two the first time)

In [ ]:
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image
t0 = time.time()
t2i = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")
i2i = AutoPipelineForImage2Image.from_pipe(t2i)
t2i.set_progress_bar_config(disable=True); i2i.set_progress_bar_config(disable=True)
W, H = 768, 432
print(f"loaded in {time.time() - t0:.0f}s")

## Encoder, visitors and the crowd-vector math

In [ ]:
@torch.no_grad()
def E(text):
    """CLIP text embedding, the same conditioning the model normally gets from a prompt. Shape (1, 77, 1024)."""
    emb, _ = t2i.encode_prompt(text, device="cuda", num_images_per_prompt=1, do_classifier_free_guidance=False)
    return emb

STYLE = ("an impressionist oil painting by Claude Monet, a gothic chapel tower above a garden pond, "
         "reflections on the water, thick visible brushstrokes")
S_EMB = E(STYLE)

# Voice tone and Laban movement efforts, as descriptor phrases (same pickers as the web demo)
TONES = {
    "whisper": "hushed, soft, dim, quiet",
    "calm": "calm, still, gentle",
    "warm": "warm golden glow, tender",
    "excited": "bright, vivid, lively",
    "shouting": "loud, turbulent, dramatic, stormy",
}
MOVES = {
    "float": "drifting, airy, floating mist",
    "glide": "smooth, serene, gliding",
    "dab": "light dabs of color, sparkling",
    "flick": "flickering, playful",
    "press": "heavy, pressing, dense",
    "wring": "twisting, wrung, tense",
    "punch": "forceful, explosive",
    "slash": "sweeping, slashing, windswept",
}
# (what they said, visual reading, tone, movement): the 10 example visitors from the web demo
VISITORS = [
    ("Finally submitted my thesis. I could sleep for a week.", "dusk settles, lilies open on still water", "calm", "float"),
    ("Everyone I love is in this room right now.", "golden light, the banks fill with people", "warm", "glide"),
    ("I am so stressed about the job market.", "a storm gathers over cold water", "shouting", "wring"),
    ("Missing my grandma tonight.", "night mist, an empty bank", "whisper", "press"),
    ("WE WON THE GAME!!", "a bright roar of color, crowds at the water", "shouting", "punch"),
    ("First snow of the year on the walk here.", "winter light on the tower, snow on the trees", "calm", "float"),
    ("Honestly no idea what I am doing with my life.", "thick fog swallows the far bank", "whisper", "wring"),
    ("Spring break in nine days!", "bright morning, the garden in full bloom", "excited", "flick"),
    ("I want to build something that matters.", "the air clears, the tower stands lit", "warm", "press"),
    ("Thinking about the people who are not here.", "lamplight in the windows, a quiet bank", "whisper", "glide"),
]

def expression(text, tone, move):
    """Encode one visitor: words plus 0.6 x tone plus 0.6 x movement, as a weighted average in CLIP space."""
    return (E(text) + 0.6 * E(TONES[tone]) + 0.6 * E(MOVES[move])) / 2.2

def novelty(e, C):
    """How different this visitor is from the crowd so far (cosine distance on the flattened embedding)."""
    return 1 - torch.nn.functional.cosine_similarity(e.flatten().float(), C.flatten().float(), dim=0).item()

def gen():
    return torch.Generator("cuda").manual_seed(7)

def step(img, cond, strength):
    """Decode: image-to-image from the previous frame. strength = how much of the canvas gets repainted."""
    return i2i(prompt_embeds=cond, image=img, strength=strength, num_inference_steps=4,
               guidance_scale=0.0, generator=gen()).images[0]

## Smoke test: opening frame plus one visitor

In [ ]:
t0 = time.time()
base = t2i(prompt_embeds=S_EMB, num_inference_steps=1, guidance_scale=0.0, width=W, height=H, generator=gen()).images[0]
print(f"opening frame: {time.time() - t0:.2f}s")
display(base)

words, reading, tone, move = VISITORS[1]
e = expression(words, tone, move)
t0 = time.time()
f1 = step(base, 0.45 * S_EMB + 0.55 * (S_EMB + 0.6 * (e - S_EMB)), 0.5)
print(f"one visitor ({words!r}): {time.time() - t0:.2f}s")
display(f1)

## Full run: 10 visitors × 3 variants

Update rule, the same as the web demo: `C ← C + α(e − C)`. Surprising visitors get a bigger `α` (pull) and a bigger `strength` (more of the canvas repainted). Novelty is scaled by the largest seen so far, because raw CLIP cosine distances are small. The printed values are there so you can tune it.

In [ ]:
def run(variant):
    C, img, frames, log, times, maxnov = S_EMB.clone(), base, [base], [], [], 1e-6
    for words, reading, tone, move in VISITORS:
        if variant == "text prompt only":
            cond, s, nov = E(f"{STYLE}, {reading}, {TONES[tone]}, {MOVES[move]}"), 0.5, None
        else:
            e = expression(words if variant == "raw words" else reading, tone, move)
            nov = novelty(e, C); maxnov = max(maxnov, nov); rel = nov / maxnov
            a, s = 0.35 + 0.45 * rel, 0.30 + 0.30 * rel
            C = C + a * (e - C)
            cond = 0.45 * S_EMB + 0.55 * C   # keep the style anchor so it stays one painting
        t = time.time(); img = step(img, cond, s); times.append(time.time() - t)
        frames.append(img); log.append((words, round(s, 2), None if nov is None else round(nov, 4)))
    return frames, log, times

In [ ]:
from PIL import Image, ImageDraw
os.makedirs("out", exist_ok=True)

def grid(frames, labels, cols=4):
    w, h = frames[0].size; tw, th = w // 2, h // 2
    rows = (len(frames) + cols - 1) // cols
    G = Image.new("RGB", (cols * tw, rows * (th + 24)), "white"); d = ImageDraw.Draw(G)
    for i, (f, lab) in enumerate(zip(frames, labels)):
        x, y = (i % cols) * tw, (i // cols) * (th + 24)
        G.paste(f.resize((tw, th)), (x, y)); d.text((x + 4, y + th + 5), lab[:62], fill="black")
    return G

results = {}
for v in ["raw words", "visual reading", "text prompt only"]:
    frames, log, times = run(v); results[v] = (frames, log, times)
    labels = ["opening"] + [f"{i + 1}. {w[:38]} s={s}" for i, (w, s, _) in enumerate(log)]
    slug = v.replace(" ", "_")
    G = grid(frames, labels); G.save(f"out/{slug}_grid.jpg", quality=90)
    frames[0].save(f"out/{slug}.gif", save_all=True, append_images=frames[1:], duration=900, loop=0)
    print(f"\n=== {v} === median {sorted(times)[len(times) // 2]:.2f}s per frame")
    for i, (w, s, nov) in enumerate(log): print(f"  {i + 1:>2}. novelty={nov}  strength={s}  {w}")
    display(G)

## Score it

For each variant, note pass or fail on the four criteria at the top, plus one line on what surprised you. Then download the results and share the grids and GIFs with the team.

In [ ]:
!zip -qr living_canvas_results.zip out
try:
    from google.colab import files
    files.download("living_canvas_results.zip")
except ImportError:
    print("Not in Colab: results are in ./out and living_canvas_results.zip")